In [ ]:
import sys, torch, platform
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print('GPU:', gpu.name, f'({gpu.total_memory / 2**30:.1f} GiB VRAM)')
assert torch.cuda.is_available(), 'Please select a GPU runtime before model loading.'

Python: 3.13.16
PyTorch: 2.11.0+cu130
CUDA: True
GPU: Tesla T4 (14.6 GiB VRAM)


In [ ]:
%pip -q install "textcraft==0.0.3" "transformers>=4.45,<5" "accelerate>=1.0,<2" "bitsandbytes>=0.45,<1"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
from pathlib import Path
from importlib import resources
from textcraft import TextCraft
import importlib.metadata

TEXTCRAFT_DATA = str(resources.files('textcraft').joinpath('data'))
assert (Path(TEXTCRAFT_DATA) / 'recipes').is_dir(), f'Recipe directory not found: {TEXTCRAFT_DATA}'

def make_env():
    return TextCraft(minecraft_dir=TEXTCRAFT_DATA)

env = make_env()
initial_obs, info = env.reset(seed=42)
print('TextCraft package:', importlib.metadata.version('textcraft'))
print('INITIAL TASK:')
print(initial_obs)
print('\n--- Non-scoring environment smoke test ---')
obs, reward, terminated, truncated, info = env.step('inventory')
print('Command: inventory')
print('Feedback:', obs)
print('Reward:', reward, 'Terminated:', terminated, 'Truncated:', truncated)
env.close()

TextCraft package: 0.0.3
INITIAL TASK:
Crafting commands:
craft 1 red sandstone using 4 red sand
craft 6 cut red sandstone slab using 3 cut red sandstone
craft 4 red sandstone stairs using 6 red sandstone
craft 4 cut red sandstone using 4 red sandstone
craft 6 red sandstone wall using 6 red sandstone
craft 6 red sandstone slab using 3 red sandstone

Goal: craft cut red sandstone slab.

--- Non-scoring environment smoke test ---
Command: inventory
Feedback: Inventory: You are not carrying anything.
Reward: 0 Terminated: False Truncated: False


In [ ]:
import json, random, re, time, csv, datetime
from pathlib import Path

USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = Path('/content/drive/MyDrive/agent_research/textcraft_7b_pilot')
else:
    OUT = Path('/content/textcraft_7b_pilot')
OUT.mkdir(parents=True, exist_ok=True)

SEED = 42
MAX_STEPS = 30
HISTORY_WINDOW = 8
MAX_NEW_TOKENS = 100
MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
PILOT_SEEDS = list(range(SEED, SEED + 5))
LARGER_SEEDS = list(range(SEED, SEED + 50))

# Create a task manifest before running any model: no task cherry-picking.
manifest_path = OUT / 'textcraft_task_manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print('Loaded existing manifest:', manifest_path)
else:
    checker = make_env()
    manifest = []
    for s in LARGER_SEEDS:
        start, _ = checker.reset(seed=s)
        manifest.append({'seed': s, 'initial_observation': start})
    checker.close()
    manifest_path.write_text(json.dumps(manifest, indent=2))
    print('Created manifest:', manifest_path)

assert [r['seed'] for r in manifest[:5]] == PILOT_SEEDS, 'Manifest seed mismatch; inspect old experiment files.'
print('Manifest entries:', len(manifest), 'Output:', OUT)

Mounted at /content/drive
Created manifest: /content/drive/MyDrive/agent_research/textcraft_7b_pilot/textcraft_task_manifest.json
Manifest entries: 50 Output: /content/drive/MyDrive/agent_research/textcraft_7b_pilot


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

torch.manual_seed(SEED)
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map='auto',
    quantization_config=quant_config,
    torch_dtype=torch.float16,
)
model.eval()
print('Model ready:', MODEL_ID)
if torch.cuda.is_available():
    print('Allocated GPU memory (GiB):', round(torch.cuda.memory_allocated() / 2**30, 2))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

... (6 outputs left out)

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model ready: Qwen/Qwen2.5-7B-Instruct
Allocated GPU memory (GiB): 5.18


In [ ]:
SYSTEM_PROMPT = """You are an agent solving a text-based Minecraft crafting task.
You are given a list of available crafting commands and a target item.
Your job is to craft the target item in as few actions as possible.

Allowed commands:
- get N ITEM          (gather N of a basic/non-craftable item)
- craft N ITEM using N INGREDIENT, N INGREDIENT  (copy an exact available crafting command)
- inventory           (inspect your inventory)

Important rules:
- First plan the dependency order of recipes, then execute one command at a time.
- Gather base ingredients before crafting the items that depend on them.
- Do not try to 'get' any item that has to be crafted.
- Use only crafting commands from the task's displayed recipe list.
- Use exact names and explicit integer quantities for get/craft commands.
- Never assume a previous command succeeded unless the environment confirmed it.
- Respond with exactly one line: Action: <one command>
- Do not add explanations, markdown, or additional actions.
"""

def build_messages(initial_task, history):
    recent = history[-HISTORY_WINDOW:]
    lines = [
        'TASK AND ALLOWED CRAFTING RECIPES:',
        initial_task,
        '',
        'RECENT ACTIONS AND ENVIRONMENT FEEDBACK:',
    ]
    if recent:
        for i, step in enumerate(recent, 1):
            lines.append(f"{i}. Action: {step['action']}")
            lines.append(f"   Feedback: {step['observation']}")
    else:
        lines.append('(No actions taken yet.)')
    lines.extend(['', 'Choose the next command.', 'Action:'])
    return [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': '\n'.join(lines)},
    ]

@torch.inference_mode()
def generate_action_text(initial_task, history):
    messages = build_messages(initial_task, history)
    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    )
    input_ids = input_ids.to(next(model.parameters()).device)
    generated = model.generate(
        input_ids=input_ids,
        attention_mask=torch.ones_like(input_ids),
        do_sample=False,
        max_new_tokens=MAX_NEW_TOKENS,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(generated[0][input_ids.shape[1]:], skip_special_tokens=True).strip()

def parse_action(raw):
    # Accept either the requested Action: prefix or a plain single command.
    matches = re.findall(r'(?im)^\s*Action\s*:\s*(.*)$', raw)
    if len(matches) == 1:
        command = matches[0].strip().strip('`')
    elif len(matches) == 0 and len(raw.splitlines()) == 1:
        command = raw.strip().strip('`')
    else:
        return None
    if re.fullmatch(r'get [1-9][0-9]* [a-z0-9 ]+', command):
        return command
    if re.fullmatch(r'craft [1-9][0-9]* [a-z0-9 ]+ using (?:[1-9][0-9]* [a-z0-9 ]+)(?:, [1-9][0-9]* [a-z0-9 ]+)*', command):
        return command
    if command == 'inventory':
        return command
    return None

assert parse_action('Action: get 4 oak log') == 'get 4 oak log'
assert parse_action('Action: craft 4 oak planks using 1 oak log') == 'craft 4 oak planks using 1 oak log'
assert parse_action('Action: inventory') == 'inventory'
print('Policy and parser ready.')

Policy and parser ready.


In [ ]:
MANIFEST_BY_SEED = {entry['seed']: entry['initial_observation'] for entry in manifest}

def run_episode(seed, verbose=True, save=True):
    env = make_env()
    initial_task, reset_info = env.reset(seed=seed)
    if seed in MANIFEST_BY_SEED and initial_task != MANIFEST_BY_SEED[seed]:
        env.close()
        raise RuntimeError(
            f'TextCraft task changed for seed {seed}. Check package/runtime versions. '
            'Do not compare this run to the stored manifest.'
        )
    history, trace = [], []
    success, truncated_by_budget = False, False
    total_reward, parser_errors, environment_errors = 0.0, 0, 0
    if verbose:
        print(f'\n===== EPISODE SEED {seed} =====')
        print(initial_task)
    started = time.time()
    try:
        for t in range(1, MAX_STEPS + 1):
            tic = time.time()
            raw = generate_action_text(initial_task, history)
            action = parse_action(raw)
            if action is None:
                parser_errors += 1
                action = '<PARSE_ERROR>'
                next_obs = 'Invalid action format. Reply with exactly one valid TextCraft command.'
                reward, terminated, truncated, info = 0.0, False, False, {}
            else:
                next_obs, reward, terminated, truncated, info = env.step(action)
                # Identify common environment-level errors, without conflating them with parse errors.
                if next_obs.startswith(('Could not', 'Wrong item')):
                    environment_errors += 1
            success = bool(terminated and reward == 1)
            total_reward += float(reward)
            record = {
                'step': t,
                'raw_model_output': raw,
                'action': action,
                'observation': next_obs,
                'reward': float(reward),
                'terminated': bool(terminated),
                'truncated': bool(truncated),
                'inference_and_step_seconds': round(time.time()-tic, 3),
            }
            trace.append(record)
            history.append({'action': action, 'observation': next_obs})
            if verbose:
                print(f'\n[{t}/{MAX_STEPS}] Action: {action}')
                print(f'   Observation: {next_obs}')
                print(f'   Reward: {reward} | Terminated: {terminated}')
                if action == '<PARSE_ERROR>':
                    print(f'   Raw model output: {raw!r}')
            if terminated or truncated:
                break
        else:
            truncated_by_budget = True
    finally:
        env.close()

    result = {
        'seed': seed,
        'model': MODEL_ID,
        'initial_task': initial_task,
        'success': success,
        'steps': len(trace),
        'reward': total_reward,
        'parser_errors': parser_errors,
        'environment_errors': environment_errors,
        'hit_step_limit': truncated_by_budget,
        'elapsed_seconds': round(time.time()-started, 2),
        'trace': trace,
    }
    if save:
        (OUT / f'textcraft_seed_{seed}.json').write_text(json.dumps(result, indent=2))
    if verbose:
        print(f'\nRESULT: success={success}, steps={len(trace)}, parser_errors={parser_errors}')
    return result

In [ ]:
one = run_episode(PILOT_SEEDS[0], verbose=True)
print('Saved trace:', OUT / f'textcraft_seed_{PILOT_SEEDS[0]}.json')

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



===== EPISODE SEED 42 =====
Crafting commands:
craft 1 red sandstone using 4 red sand
craft 6 cut red sandstone slab using 3 cut red sandstone
craft 4 red sandstone stairs using 6 red sandstone
craft 4 cut red sandstone using 4 red sandstone
craft 6 red sandstone wall using 6 red sandstone
craft 6 red sandstone slab using 3 red sandstone

Goal: craft cut red sandstone slab.

[1/30] Action: get 4 red sand
   Observation: Got 4 red sand
   Reward: 0 | Terminated: False

[2/30] Action: craft 1 cut red sandstone using 4 red sand
   Observation: Crafted 1 minecraft:red_sandstone
   Reward: 0 | Terminated: False

[3/30] Action: get 4 red sand
   Observation: Got 4 red sand
   Reward: 0 | Terminated: False

[4/30] Action: get 3 red sandstone
   Observation: Could not find red sandstone
   Reward: 0 | Terminated: False

[5/30] Action: get 3 red sandstone
   Observation: Could not find red sandstone
   Reward: 0 | Terminated: False

[6/30] Action: get 3 red sandstone
   Observation: Could not 

In [ ]:
# Rebuild the manifest with a fresh environment for each seed.
# Use a new filename so we preserve the original manifest.

manifest_path = OUT / "textcraft_task_manifest_v2.json"

manifest = []

for seed in LARGER_SEEDS:
    task_env = make_env()
    try:
        observation, _ = task_env.reset(seed=seed)
        manifest.append({
            "seed": seed,
            "initial_observation": observation
        })
    finally:
        task_env.close()

manifest_path.write_text(json.dumps(manifest, indent=2))

# Update the lookup used by run_episode()
MANIFEST_BY_SEED = {
    entry["seed"]: entry["initial_observation"]
    for entry in manifest
}

# Check that fresh environments reproduce the pilot tasks.
for seed in PILOT_SEEDS:
    check_env = make_env()
    try:
        observation, _ = check_env.reset(seed=seed)
        assert observation == MANIFEST_BY_SEED[seed], (
            f"Task still differs for seed {seed}"
        )
    finally:
        check_env.close()

print("Manifest rebuilt and pilot seeds verified.")

Manifest rebuilt and pilot seeds verified.


In [ ]:
pilot = [run_episode(s, verbose=False) for s in PILOT_SEEDS]
for r in pilot:
    print(f"seed={r['seed']:>3} | success={str(r['success']):<5} | steps={r['steps']:>2} | parse_errors={r['parser_errors']}")
print(f"Pilot success: {sum(r['success'] for r in pilot)}/{len(pilot)} = {sum(r['success'] for r in pilot)/len(pilot):.1%}")

seed= 42 | success=True  | steps=29 | parse_errors=0
seed= 43 | success=False | steps=30 | parse_errors=0
seed= 44 | success=False | steps=30 | parse_errors=5
seed= 45 | success=False | steps=30 | parse_errors=0
seed= 46 | success=False | steps=30 | parse_errors=8
Pilot success: 1/5 = 20.0%


In [ ]:
for r in pilot:
    print(f"\n{'='*60}")
    print(f"SEED {r['seed']} | SUCCESS: {r['success']}")
    print(f"Steps: {r['steps']}")
    print(f"Parse errors: {r['parser_errors']}")
    print(f"Environment errors: {r['environment_errors']}")

    print("\nLAST 8 ACTIONS:")
    for step in r["trace"][-8:]:
        print(f"\nStep {step['step']}: {step['action']}")
        print(f"Feedback: {step['observation'][:250]}")

    errors = [
        step for step in r["trace"]
        if step["action"] == "<PARSE_ERROR>"
    ]

    if errors:
        print("\nFIRST 3 RAW PARSING FAILURES:")
        for step in errors[:3]:
            print(repr(step["raw_model_output"]))


SEED 42 | SUCCESS: True
Steps: 29
Parse errors: 0
Environment errors: 16

LAST 8 ACTIONS:

Step 22: get 1 red sandstone
Feedback: Could not find red sandstone

Step 23: get 4 red sandstone
Feedback: Could not find red sandstone

Step 24: get 4 red sandstone
Feedback: Could not find red sandstone

Step 25: get 4 red sand
Feedback: Got 4 red sand

Step 26: craft 4 red sandstone using 4 red sand
Feedback: Crafted 1 minecraft:red_sandstone

Step 27: get 4 red sand
Feedback: Got 4 red sand

Step 28: craft 3 cut red sandstone using 4 red sandstone
Feedback: Crafted 4 minecraft:cut_red_sandstone

Step 29: craft 6 cut red sandstone slab using 3 cut red sandstone
Feedback: Crafted 6 minecraft:cut_red_sandstone_slab

SEED 43 | SUCCESS: False
Steps: 30
Parse errors: 0
Environment errors: 30

LAST 8 ACTIONS:

Step 23: craft 3 yellow wool using 1 yellow dye, 1 white wool
Feedback: Could not find enough items to craft minecraft:yellow_wool

Step 24: craft 3 yellow wool using 1 yellow dye, 1 white w

In [ ]:
from collections import Counter
import re
import json

# Preserve V1 results and output files
pilot_v1 = pilot
OUT_V1 = OUT

OUT = OUT_V1.parent / "textcraft_7b_pilot_v2"
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "task_manifest.json").write_text(
    json.dumps(manifest, indent=2)
)

# 1. Fix command parsing
parse_action_v1 = parse_action

def parse_action(raw):
    cleaned = re.sub(r"(?<=\w)_(?=\w)", " ", raw)
    action = parse_action_v1(cleaned)

    # 'using' is only legal in craft commands
    if action and action.startswith("get ") and " using " in action:
        return None

    return action

# 2. Strengthen the agent prompt
SYSTEM_PROMPT += """

Additional rules:
- 'get' is only for base materials, never for crafting.
- 'using' must never appear in a get command.
- If a crafting action fails, identify which ingredients
  are missing before trying it again.
- Do not repeat a failed action unless something relevant
  has changed, such as obtaining additional materials.
- Use inventory to verify your resources when uncertain.
- Follow the quantities in the provided recipes.
"""

# 3. Keep track of failures beyond the history window
build_messages_v1 = build_messages

def build_messages(initial_task, history):
    messages = build_messages_v1(initial_task, history)

    failures = Counter(
        step["action"]
        for step in history
        if step["observation"].startswith(
            ("Could not", "Wrong item")
        )
    )

    if failures:
        summary = [
            f"- {action}: failed {count} time(s)"
            for action, count in failures.most_common(10)
        ]

        messages[1]["content"] += (
            "\n\nPREVIOUS FAILED ACTIONS:\n"
            + "\n".join(summary)
            + "\nReassess dependencies before retrying. "
              "An action may become possible after obtaining "
              "the necessary ingredients."
        )

    return messages

print("V2 policy ready.")
print("V1 results preserved at:", OUT_V1)
print("V2 output folder:", OUT)

V2 policy ready.
V1 results preserved at: /content/drive/MyDrive/agent_research/textcraft_7b_pilot
V2 output folder: /content/drive/MyDrive/agent_research/textcraft_7b_pilot_v2


In [ ]:
pilot_v2 = [
    run_episode(s, verbose=False)
    for s in PILOT_SEEDS
]

print("Seed | V1 success | V2 success | V2 env errors")

for old, new in zip(pilot_v1, pilot_v2):
    print(
        old["seed"],
        old["success"],
        new["success"],
        new["environment_errors"]
    )

print(
    "V2 success:",
    sum(r["success"] for r in pilot_v2),
    "/",
    len(pilot_v2)
)

Seed | V1 success | V2 success | V2 env errors
42 True False 16
43 False False 22
44 False False 18
45 False False 29
46 False False 15
V2 success: 0 / 5
